In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import warnings
from transformers import AutoModelForCausalLM, AutoProcessor
import torch

base_path = "/inspire/hdd/global_user/240108540141/models/PA-BDM"

model = AutoModelForCausalLM.from_pretrained(
    base_path,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    trust_remote_code=True,
)

model = model.to("cuda")
model.eval()
processor = AutoProcessor.from_pretrained(
    base_path,
    trust_remote_code=True,
)

/root/anaconda3/envs/diffusionvl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.05s/it]


In [ ]:
from PIL import Image
import time


# 1.text
# image = Image.open("./example_text.jpg")
# messages = [
#     {"role": "user", "content": [
#         {"type": "image"},
#         {"type": "text", "text": "<image>\nText Recognition."}
#     ]}
# ]


# 2.formula
image = Image.open("./example_formula.jpg")
messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "<image>\nFormula Recognition."}
    ]}
]


# 3.table
# image = Image.open("./example_table.jpg")
# messages = [
#     {"role": "user", "content": [
#         {"type": "image"},
#         {"type": "text", "text": "<image>\nTable Recognition."}
#     ]}
# ]



text = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)
inputs = {k: v.to(model.device) if hasattr(v, 'to') else v for k, v in inputs.items()}

# Generate with diffusion
t1 = time.time()
output_ids = model.generate(
    inputs=inputs["input_ids"],
    images=inputs.get("pixel_values"),
    image_grid_thws=inputs.get("image_grid_thw"),
    gen_length=1024,
    steps=32,   # block_size
    temperature=0.0,
    confidence_threshold=0.95, # confidence_threshold
    without=None, 
)
t = time.time() - t1
print(t)

# Decode output
output_text = processor.decode(output_ids[0], skip_special_tokens=False).replace('<|im_end|>','')

0.41228222846984863


In [19]:
print(output_text)

The dynamics of the sample temperature are governed by a group of processes, which include the radiation absorption, heat propagation into the material bulk and to the sample surface, and, potentially, to the phase transitions of the sample material. While the first process is responsible for the heating of the sample material, the other ones lead to the sample cooling. In order to construct the model of heat transfer in the target, we assess the importance of each channel of energy redistribution in the material.



In [20]:
print(len(processor.tokenizer.encode(output_text))/t)

522.8364409337235
